In [59]:
import pandas as pd
import numpy as np
import math
import re

In [ ]:
df = pd.read_csv("spam.csv", encoding='latin-1')
print(df)


# v1 = label (spam/ham), v2 = text message
y = df['v1'].values  # labels
x = df['v2'].values  # raw text messages

rows = len(x)

# Shuffle and split
np.random.seed(42)  
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x[train_idx]  
x_test = x[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

In [ ]:
#A map that stores words in the format word:index
vocab_list = {}
#len(vocab_list)
vocab_size = 100

#A simple tokenization of the text
def tokenize(text):
    text = text.lower()                   
    text = re.sub(r'[^a-z\s]', '', text)  
    tokens = text.split()                  
    return tokens

#Creating tokens for training and tetsing which will later be vectorized
train_tokens = [tokenize(x_train[i]) for i in range(len(x_train))]
test_tokens = [tokenize(x_test[j]) for j in range(len(x_test))]

#Creating the vocablist using vocab_size words based on the highest occuring 
#word count
def build_vocab_list(items):
    counter_map = {}
    #Storing a counter of each word to keep the max vocab_size words occuring
    for i in range(len(x_train)):
        for token in train_tokens[i]:
            if token not in counter_map:
                counter_map[token] = 1
            else:
                counter_map[token] += 1
    
    sorted_map = sorted(counter_map.items(), key=lambda item:item[1], reverse=True)
    top_words = sorted_map[:items]

    final_map = {}
    counter = 0
    for element in top_words:
        word, number = element
        final_map[word] = counter
        counter += 1
    
    return final_map

vocab_list = build_vocab_list(vocab_size)

#Creating vectorized inputs for each training example in x_train
vectorized_inputs = [[0] * vocab_size for _ in range(len(x_train))]
def vectorize(input_matrix):
    vectorized_inputs = [[0] * vocab_size for _ in range(len(input_matrix))]
    for i in range(len(input_matrix)):
        for token in train_tokens[i]:
            if token in vocab_list:
                vectorized_inputs[i][vocab_list[token]] = 1
        
    return vectorized_inputs

vectorized_inputs = vectorize(x_train)

#Calculate class ratios (simple count of each class)
class_ratio_map = {}
def calculate_class_ratio(training_y):
    for i in range(len(training_y)):
        if training_y[i] not in class_ratio_map:
            class_ratio_map[training_y[i]] = 1
        else:
            class_ratio_map[training_y[i]] += 1


calculate_class_ratio(y_train)
print(class_ratio_map)

In [ ]:

#This will store the phi parameter used in the Bernoulli Distribution as (k, j) where k corresponds to the class and j corresponds to the feature
phi_dict = {}

#phi j,k = (occurences of a feature exisiting given a specific class)/(number of examples of that specific class)
def calculate_phi_dict(vectorized_x, training_y):
    feature_number = 0
    for i in range(len(vectorized_x[0])):
        auxilary_dict = {element:0 for element in class_ratio_map}
        for j in range(len(vectorized_x)):
            auxilary_dict[training_y[j]] += vectorized_x[j][i]
    
        for elt in auxilary_dict:
            #Laplace smoothing to ensure log(0) probabilities never appear
            phi_dict[(elt, feature_number)] = (auxilary_dict[elt] + 1)/(class_ratio_map[elt] + 2)
        
        feature_number += 1


calculate_phi_dict(vectorized_inputs, y_train)

#Bernoulli Distribution function
def bernoulli_dist(x, phi):
    return ((phi)**x) * ((1-phi)**(1-x))

#Gives prediction for an input test vector. Using the Naive Bayes assumption that each feature is independant
#meaning P(X,Y) = P(X|Y) * P(Y) = ∏P(Xj|Y) * P(Y). Applying log probabilities: log(∏P(Xj|Y) * P(Y)) = 
#log(∑P(Xj|Y)) + log(P(Y)). Using the bernoulli distribution and the phi_dict calculated, plug in
#values for P(Xj|Y) and P(Y) for each class and return the prediction as the one which corresponds to the
#highest log likelihood
def predict_one(test_vector, training_y):
    prediction = ''
    prediction_dict = {elt:math.log(class_ratio_map[elt]/len(training_y)) for elt in class_ratio_map}
    for i in range(len(test_vector)):
        for element in class_ratio_map:
            prediction_dict[element] += math.log(bernoulli_dist(test_vector[i], phi_dict[(element, i)]))
    
    num = float('-inf')
    for element in prediction_dict:
        if (prediction_dict[element]) > num:
            num = prediction_dict[element]
            prediction = element
    
    return prediction
    

#Vectorizing test inputs
vectorized_test_inputs = [[0] * 1000 for _ in range(len(x_test))]
vectorized_test_inputs = vectorize(x_test)

#Making predictions
def make_all_predictions(test_x, test_y, training_y):
    accuracy = 0
    ham_count = 0
    spam_count = 0
    for i in range(len(test_x)):
        prediction = predict_one(test_x[i], training_y)
        if(prediction == test_y[i]):
            accuracy += 1

        if prediction == 'ham':
            ham_count += 1
        else:
            spam_count += 1
    
    print(accuracy/len(test_x))

make_all_predictions(vectorized_test_inputs, y_test, y_train)
